## Mission and team roles

Produce a causal account of a compiler integrity violation, a small reproducer, and a verification gate that detects compromised outputs without rejecting legitimate rewrites.

| Workstream | Suggested owner | First task |
|---|---|---|
| Investigator | Rotate from Challenge 1 | Form hypotheses and compare incident cases |
| Exploit developer | Rotate from Challenge 1 | Discover activating inputs and minimize a witness |
| Defense analyst | Rotate from Challenge 1 | Design tests, characterize false positives, record limitations |

Responsibilities are flexible for one- or two-person teams. After the first 60–90 minutes, exchange findings: each member should explain **another member's result**. Do not leave documentation and presentation to the final few minutes.

**Suggested pacing:** 45 min orientation/triage; 75 min localization; 60 min trigger/reproducer; 60 min defense; 30–60 min extension and submission. Prioritize a defensible core over exhaustive bonus work.

## Threat model and contract

```text
Trusted application / source circuit
                │
                ▼
       compiler service under review
       ├─ normalization
       ├─ phase convention
       ├─ vendor plugin
       └─ native-basis lowering
                │
                ▼
   candidate circuit → trusted local simulator → observations
```

Assume the source circuit, local Qiskit math routines, and local simulator are trusted. The attacker controls a third-party compilation plugin, not the source, verifier, simulator, operating system, or your notebook. Source access is allowed: inspection is a valid investigative path, but it does not replace experiments or generalization.

**Lab contract:** bound measurement-free unitary circuits, 1–6 qubits; no classical bits, dynamic control, ancillas, physical placement, or routing. Input and output logical wires have the same meaning. The compiler promises exact standalone-unitary equivalence **up to global phase**, within numerical tolerance. Basis gates are `rz`, `sx`, `x`, and `cx`. Do not infer production scalability from this deliberately small model.

A global phase is acceptable under this standalone-circuit contract. It may become observable if the whole circuit is later coherently controlled; that use case is outside this contract. Physical-layout-aware comparison is also outside this exercise.

In [4]:
from pathlib import Path
import csv
import json
import platform
from importlib.metadata import version

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector, state_fidelity

from breaq_ch2 import (
    CompilerService, exact_probabilities, sample_counts,
    total_variation, circuit_digest,
)

service = CompilerService("team_instance.json")
workloads = service.workloads()
print("Team:", service.team_id)
print("Python:", platform.python_version(), "Qiskit:", version("qiskit"))
print("Workloads:", list(workloads))
print(json.dumps(service.contract, indent=2))

Team: DEMO-TEAM
Python: 3.12.12 Qiskit: 2.2.3
Workloads: ['CAL-BELL', 'LEGACY-REDUNDANCY', 'SCAN-A', 'SCAN-B', 'SCAN-C', 'SCAN-D', 'SCAN-E', 'SCAN-F']
{
  "basis_gates": [
    "rz",
    "sx",
    "x",
    "cx"
  ],
  "max_qubits": 6,
  "routing": "disabled / logical wire order preserved",
  "equivalence": "standalone unitary, up to global phase",
  "approximate_synthesis": false
}


## API reference

| API | Result | Use |
|---|---|---|
| `service.workloads()` | Dictionary of circuit copies | Incident inputs |
| `service.compile(qc, build="suspect")` | Candidate circuit | Service under review |
| `service.compile(qc, build="baseline")` | Candidate circuit | Trusted baseline under the stated threat model |
| `service.trace(qc, build="suspect")` | Stage dictionaries with `circuit`, `depth`, `operations` | Local diagnostic snapshots |
| `exact_probabilities(qc)` | Displayed-bit-string → probability | Exact computational-basis distribution from all-zero input |
| `sample_counts(qc, shots=1024, seed=2026)` | Displayed-bit-string → count | Finite-shot ideal local observations |
| `total_variation(p, q)` | Float in `[0, 1]` | Compare **normalized** distributions |
| `circuit_digest(qc)` | SHA-256 of QPY bytes | Artifact identification, **not semantic verification** |

The service never returns an “infected” label. Trace names, metadata, and self-reported statistics are not trust anchors. The local trace API is an educational convenience, not proof that a remote service executed the same pipeline.

## Warm-up: legitimate rewriting

The two circuits below implement the same standalone operation but have different gate sequences. Add a benign phase convention as well. Explain what a verifier should accept.

Qiskit passes operate on a DAG representation; transformations may change syntax without changing meaning. `Operator.equiv` compares operators up to global phase [IBM transpiler documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/transpiler), [IBM Operator API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/qiskit.quantum_info.Operator).

In [8]:
original = QuantumCircuit(1)
original.h(0)
rewritten = QuantumCircuit(1)
rewritten.rz(np.pi / 2, 0)
rewritten.sx(0)
rewritten.rz(np.pi / 2, 0)
rewritten.global_phase += np.pi / 9
print(original.draw(output="text"))
print(rewritten.draw(output="text"))
print("Equivalent up to global phase:", Operator(original).equiv(Operator(rewritten)))

   ┌───┐
q: ┤ H ├
   └───┘
global phase: π/9
   ┌─────────┐┌────┐┌─────────┐
q: ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├
   └─────────┘└────┘└─────────┘
Equivalent up to global phase: True


## Level 1 — Incident triage

First reproduce sampled observations. Select controls as well as suspicious cases. A distribution from one input and one measurement basis does **not** characterize an entire unitary. Conversely, different finite-shot histograms alone do not prove an integrity violation.

In [9]:
compiled = {name: service.compile(qc) for name, qc in workloads.items()}
triage_observations = {}
for name in ("CAL-BELL", "SCAN-B", "SCAN-D"):
    triage_observations[name] = {
        "source_counts": sample_counts(workloads[name], shots=512),
        "candidate_counts": sample_counts(compiled[name], shots=512),
    }
    print(name, triage_observations[name])

CAL-BELL {'source_counts': {'00': 261, '11': 251}, 'candidate_counts': {'00': 261, '11': 251}}
SCAN-B {'source_counts': {'100': 196, '111': 190, '101': 65, '110': 61}, 'candidate_counts': {'101': 206, '110': 194, '111': 57, '100': 55}}
SCAN-D {'source_counts': {'100': 123, '110': 129, '101': 138, '111': 122}, 'candidate_counts': {'100': 123, '110': 129, '101': 138, '111': 122}}


In [ ]:
# Your experiment: compare more inputs or exact distributions.
# Record the difference between an observation and a security conclusion.
triage_evidence = []
initial_hypotheses = []

### Core investigation

Create a table for **every** workload containing at least:

- Source and compiled gate counts/depth.
- Exact computational-basis total variation distance.
- State fidelity from the all-zero input.
- Full-unitary equivalence up to global phase.

Predict at least one disagreement among those metrics before computing them. Why might a histogram-only or single-input-state checker be insufficient?

In [ ]:
# Implement your own comparison, not a workload-name lookup.
def inspect_compilation(source, candidate):
    # Return a JSON-serializable dictionary of evidence.
    return {"status": "not implemented"}

comparison_rows = []
for name, source in workloads.items():
    comparison_rows.append({"case_id": name, **inspect_compilation(source, compiled[name])})
comparison_rows

### Threat-model worksheet

Fill this in before looking for implementation details. Which claim is stronger: “same observed output distribution” or “same standalone unitary”? What is actually required by the service contract?

In [ ]:
threat_model = {
    "protected_asset": "",
    "attacker_capability": "",
    "trust_boundary": "",
    "security_property": "",
    "trusted_components": [],
    "out_of_scope": [],
}

## Level 2 — Locate the first violation

Obtain a detached snapshot after each compilation stage. Determine the **first transition** that changes semantics. Do not equate the first syntax change, first phase change, or first reduction in gate count with a violation.

Use source-versus-stage checks or consecutive-stage checks. State whether your conclusion depends on trusting the trace instrumentation.

In [3]:
focus = "SCAN-B"
trace = service.trace(workloads[focus])
for snapshot in trace:
    print(snapshot["stage"], "depth=", snapshot["depth"], "ops=", snapshot["operations"])

# Available for inspection, but no fault labels are supplied:
# print(trace[index]["circuit"].draw(output="text"))

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2232257843.py, line 6)

In [ ]:
stage_evidence = []
first_bad_transition = ""
# TODO: identify the first semantic failure, with a metric and counterexample.

### Trigger discovery

Form and test a hypothesis about **when** the service changes semantics. Change one feature at a time: parameters, gate positions, qubit roles, spectators, and input-state preparation. Try both predicted trigger and predicted non-trigger cases.

Source inspection is permitted. If used, cite the relevant logic and then verify the inferred condition experimentally. The notebook must stand on evidence even when a jury instance uses different numeric parameters.

In [ ]:
trigger_hypothesis = ""
trigger_experiments = []
# TODO: construct and compile your own circuits; vary parameters or structure.

### Hint ladder — optional

Hints are visible for practice. For event-time controlled release, the organizer can remove this cell and release the same text separately. Self-reported hint use is not an enforceable scoring mechanism.

<details><summary>Hint 1 — Comparing meanings</summary>
A valid basis translation can replace an entire gate sequence. Compare the promised operation, not its spelling or its length.
</details>

<details><summary>Hint 2 — Observability</summary>
Relative phases can matter even when a computational-basis histogram is unchanged. Test a different input or add a trusted analysis basis **after** compilation.
</details>

<details><summary>Hint 3 — Fault localization</summary>
Compare unitary operators at adjacent trace stages; skip global-phase differences allowed by the contract.
</details>

<details><summary>Hint 4 — Counterfactual experiment</summary>
Compare the two compiler builds on related parameterized circuits. Track whether changing a rotation's location or direction changes activation.
</details>

## Minimal reproducer and observable witness

Produce two related artifacts:

1. A small source circuit for which the suspect compiler violates the equivalence contract.
2. A trusted preparation and analysis experiment that makes the difference observable.

A tiny reproducer need not show a difference on all-zero input. The preparation and analysis may be outside the submitted subroutine; make that distinction explicit. Record your minimization criterion and which deletions you tried. Renaming a supplied workload is not minimization.

In [ ]:
minimal_source = None       # Replace with your own QuantumCircuit.
reproducer_evidence = {}
# TODO: preserve evidence, not a precomputed flag.

## Level 3 — Verification gate

Implement `verify_compilation(source, candidate) -> bool`:

- Return `True` only for equivalent in-contract circuits, up to global phase.
- Return `False` for a semantic mismatch, unsupported input, incompatible width, or failed validation.
- Do not trust circuit names, workload IDs, metadata, counts, stage labels, or byte hashes.
- Respect the 6-qubit bound; do not claim a full-matrix method scales to useful large circuits.

Then implement a wrapper that obtains a candidate, verifies it independently, and either returns it or **refuses execution**. Removing one known-bad gate or checking a specific trigger is not a general verifier.

This is small-system numerical translation validation, not a formal proof or a scalable defense for arbitrary quantum programs.

In [ ]:
def verify_compilation(source, candidate) -> bool:
    # TODO: fail closed until your independent verifier is implemented.
    return False


def guarded_compile(compiler, source, build="suspect"):
    candidate = compiler.compile(source, build=build)
    if not verify_compilation(source, candidate):
        raise ValueError("Compilation refused: integrity not established")
    return candidate

In [ ]:
# TODO: test good rewrites, global phases, semantic mutations, and invalid inputs.
# Report TP/FN for mismatches and TN/FP for legitimate equivalent rewrites.
# The positive class is "non-equivalent candidate".
defense_evidence = {}
verifier_limitations = []

## Advanced extensions

Choose one or more; none blocks the core submission.

- **Trigger boundary:** sweep an angle and infer an activation interval. Report scan resolution instead of claiming an exact boundary from a coarse grid.
- **Semantic metamorphism:** compare logically equivalent submitted syntaxes and show whether activation is syntax-sensitive. Explain why this is evidence, not a robust defense by itself.
- **DAG-aware pattern:** test harmless spectator operations or barriers. Distinguish adjacency on a target wire from adjacency in a printed instruction list.
- **Measurement witnesses:** show a case with equal computational-basis distributions but inequivalent unitaries, then reveal the change through trusted preparation/analysis.
- **Gate-count counterexample:** quantify why shallow/depth-reduced output is not an integrity certificate.
- **Scalability:** propose bounded local checks or a more scalable equivalence method, and explain what adversaries it may miss. Implementations are optional; do not imply a heuristic proves equivalence.

In [ ]:
extension_evidence = {}

## Evidence export

Your own measurements belong in reproducible evidence files, not screenshots alone. Save tabular results as CSV; if you create a plot, save it as PNG from the **same** recorded experimental data. Label all results “local simulation.” No example attack curve or precomputed answers are supplied.

Suggested fields: compiler build, angle, width, target/control role, TV distance, state fidelity, unitary equivalent, shots/seed, and relevant assumptions. A digest records an artifact, not its correctness.

In [ ]:
# Optional helper: call only after constructing nonempty uniform rows.
def save_evidence_rows(rows, filename="comparison.csv"):
    if not rows:
        raise ValueError("No experimental rows to export")
    destination = Path("submission_artifacts")
    destination.mkdir(exist_ok=True)
    with (destination / filename).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

# Example after your experiments: save_evidence_rows(comparison_rows)

## Claim sheet and contributions

For each claim, identify the actual experiment, a trusted assumption, and a limitation. A detection on the public suite does not prove absence of future compiler triggers.

All members should understand the fault location and at least one limitation. Contributions can be complementary; the smallest team is not penalized for combining roles.

In [ ]:
security_claim = ""
claim_sheet = []  # Dictionaries with claim, evidence, assumption, limitation.
member_contributions = []  # Names/aliases and concrete contributions.
hints_used = []           # Practice bookkeeping only.

## Submission contract

Submit this completed notebook, `submission.json`, `team_instance.json`, any evidence files, and `submission_artifacts/student_api.py` containing your standalone verifier. The Python module must expose exactly `verify_compilation(source, candidate) -> bool`; it must not import this notebook or case labels. Imports from Qiskit/NumPy are allowed. Copy your final implementation into that file manually.

Checkpoints: **Core:** comparison and fault localization; **Exploit:** unseen activating circuit plus a witness; **Defense:** general verifier, wrapper, and negative tests. Partial submissions are welcome. The export records self-reported progress, not awarded points.

In [ ]:
# Leave False until ready; a clean scaffold run must not export a false submission.
EXPORT_SUBMISSION = False
completion_status = {"core": False, "exploit": False, "defense": False}

if EXPORT_SUBMISSION:
    if not security_claim.strip() or not claim_sheet or not member_contributions:
        raise ValueError("Fill the security claim, claim sheet, and contributions first")
    destination = Path("submission_artifacts")
    destination.mkdir(exist_ok=True)
    if completion_status["defense"] and not (destination / "student_api.py").exists():
        raise ValueError("Save your standalone verifier in submission_artifacts/student_api.py")
    if minimal_source is not None:
        from qiskit import qpy
        with (destination / "minimal_source.qpy").open("wb") as handle:
            qpy.dump(minimal_source, handle)
    payload = {
        "schema": "breaq-submission-v1", "challenge": "02-treacherous-transpiler",
        "team_id": service.team_id, "instance_seed": service.seed,
        "environment": {"python": platform.python_version(), "qiskit": version("qiskit")},
        "security_claim": security_claim, "threat_model": threat_model,
        "first_bad_transition": first_bad_transition,
        "trigger_hypothesis": trigger_hypothesis,
        "completion_status": completion_status,
        "claims": claim_sheet, "member_contributions": member_contributions,
        "hints_used": hints_used, "verifier_limitations": verifier_limitations,
        "comparison": comparison_rows, "defense_evidence": defense_evidence,
        "reproducer_evidence": reproducer_evidence,
    }
    serialized = json.dumps(payload, indent=2, allow_nan=False)
    Path("submission.json").write_text(serialized, encoding="utf-8")
    print("Exported submission.json; include the notebook and evidence directory as well.")
else:
    print("Scaffold mode: no submission exported.")

## Oral-defense preparation

Be prepared to predict a jury perturbation **before** running it:

- Why can two different-looking compiled circuits be equivalent?
- Why are a matching Z-basis histogram and a matching all-zero output state insufficient in general?
- Which pass transition first violated the contract? Is an untrusted trace an attestation?
- What input features activate your exploit, and which counterexamples support that claim?
- Does a relative sign change necessarily change every experiment?
- How does your verifier handle global phase, unsupported circuits, or a wider circuit?
- Would your comparison remain valid after physical qubit remapping or introducing ancillas?
- What changes if the adversary controls the simulator or the verification environment as well?

**Presentation target:** one diagram, one controlled reproducer, one quantitative result, one mitigation, and one honest limitation.

## Research and API provenance

The lab uses an original, selective parameter-tampering plugin. The broader untrusted-compiler/gate-tampering threat model is inspired by Rupshali Roy, Subrata Das, and Swaroop Ghosh, *Hardware Trojans in Quantum Circuits, Their Impacts, and Defense* (2024), [arXiv:2402.01552](https://arxiv.org/abs/2402.01552). The lab does **not** reproduce that paper's specific attacks, dataset, or numerical performance.

Technical documentation:

- [Qiskit 2.2 transpiler: DAGs, pass managers, and stages](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/transpiler)
- [Operator: equivalence up to global phase and layout-aware construction](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/qiskit.quantum_info.Operator)
- [BasicSimulator for small local experiments](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.0/qiskit.providers.basic_provider.BasicSimulator)

The trusted reference and numerical equivalence check are educational privileges for circuits this small; their availability is not assumed for production-scale arbitrary algorithms.